In [44]:
import urllib.request, zipfile, os
import numpy as np
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity

# 1. **Load the Data**:

Using the MovieLens 100K Dataset.
i) Data sample is already clean, so we won't need to account for null values
ii) contains 1-5 star ratings.


In [45]:
if not os.path.exists('ml-100k/u.data'):
    urllib.request.urlretrieve('https://files.grouplens.org/datasets/movielens/ml-100k.zip', 'ml-100k.zip')
    zipfile.ZipFile('ml-100k.zip').extract('ml-100k/u.data')

df = pd.read_csv('ml-100k/u.data', sep='\t',         #MovieLens 100K is a tab separated file
                      names=['user_id', 'movie_id', 'rating', 'timestamp'],       #the original u.data file doesnt have a header at top. so this gives them header names
)

In [46]:
print(df.shape)
df.head()

(100000, 4)


,user_id,movie_id,rating,timestamp
0,196,242,3,881250949
1,186,302,3,891717742
2,22,377,1,878887116
3,244,51,2,880606923
4,166,346,1,886397596


In [47]:
df.info()
df.isna().sum()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 4 columns):
 #   Column     Non-Null Count   Dtype
---  ------     --------------   -----
 0   user_id    100000 non-null  int64
 1   movie_id   100000 non-null  int64
 2   rating     100000 non-null  int64
 3   timestamp  100000 non-null  int64
dtypes: int64(4)
memory usage: 3.1 MB


,0
user_id,0
movie_id,0
rating,0
timestamp,0


# 2. **Train Test Split**:

Split the data in 80-20 before performing EDA, to prevent data-leakage.
We split the data such that we use a Temporal Split- train on older ratings, test on newer ratings -> We require the timestamps to be arranged in ascending order

In [48]:
df = df.sort_values('timestamp').reset_index(drop=True)

In [49]:
split_idx = int(len(df) * 0.8)

train_df = df.iloc[:split_idx].copy()
test_df = df.iloc[split_idx:].copy()

In [50]:
#Dropping timestamp column
train_df = train_df.drop(columns=['timestamp'])
test_df = test_df.drop(columns=['timestamp'])

print(f"Train size: {len(train_df)} rows | Test size: {len(test_df)} rows")

Train size: 80000 rows | Test size: 20000 rows


Cons of a time-based (Temporal) Split of train-test: If a user only rated movies during the final 20% of the timeline, they will only be in the test set, creating a "cold start" user problem for matrix factorization

# 3. **EDA**:

Tackling the problem of User Behaviour. Some users are harsher critics(eg: give 1 star to a movie), some users are more easygoing(eg: give 4 stars to the same movie). THUS, we focus on deviation of a user from the average of ratings given by them.


In [63]:
global_train_mean = train_df['rating'].mean()

# Calculate user mean and subtract it directly
train_df['rating_centered'] = train_df['rating'] - train_df.groupby('user_id')['rating'].transform('mean')

# Save the user means in a separate variable for later use during prediction
train_user_means = train_df.groupby('user_id')['rating'].mean()

print("--- Data Centered Safely ---")
print(train_df[['user_id', 'movie_id', 'rating', 'rating_centered']].head())


--- Data Centered Safely ---
   user_id  movie_id  rating  rating_centered
0      259       255       4         0.130435
1      259       286       4         0.130435
2      259       298       4         0.130435
3      259       185       4         0.130435
4      259       173       4         0.130435


# 3. **Memory-Based Approach**:

We map our long dataframe into a wide matrix. The unobserved interactions are filled with 0 (since the mean is subtracted, 0 represents the neutral baseline).

Changing into a User-Item Matrix

In [61]:
# 1. Map IDs to matrix indices to ensure no out-of-bounds indexing
unique_users = sorted(df['user_id'].unique())
unique_movies = sorted(df['movie_id'].unique())

# Mapping user ID to matrix row number(0-based indexing)
user_to_idx = {uid: i for i, uid in enumerate(unique_users)}
movie_to_idx = {mid: i for i, mid in enumerate(unique_movies)}

n_users = len(unique_users)
n_movies = len(unique_movies)

# Create Empty matrices
R_train = np.zeros((n_users, n_movies))
R_test = np.zeros((n_users, n_movies))

# Fill training matrix with centered ratings
for row in train_df.itertuples():           # goes through every single row
    u_idx = user_to_idx[row.user_id]        # finds user's matrix row
    m_idx = movie_to_idx[row.movie_id]        # finds user's matrix column
    R_train[u_idx, m_idx] = row.rating_centered         # put the rating into the matrix

# Fill test matrix with original ratings (for evaluation)
for row in test_df.itertuples():
    u_idx = user_to_idx[row.user_id]
    m_idx = movie_to_idx[row.movie_id]
    R_test[u_idx, m_idx] = row.rating


Implement User-User Cosine

In [65]:
from sklearn.metrics.pairwise import cosine_similarity

def predict_memory_based(R_train, train_user_means, global_mean):         # Global mean is a fallback if we don't have a particular user's mean
    # Find Cosine Similarity between users
    user_sim = cosine_similarity(R_train)

    # Avoid self-similarity bias by setting diagonal to 0
    np.fill_diagonal(user_sim, 0)

    # Keep absolute values for normalizing denominators
    eps = 1e-9
    user_sim_abs = np.abs(user_sim)

    # Predict deviations: (Sim dot R) / Sum of absolute similarities
    predicted_deviations = np.dot(user_sim, R_train) / (np.dot(user_sim_abs, (R_train != 0).astype(float)) + eps)       # We create a mask of 0,1, which tells if rating-exists

    # Getting our Original scale back, by adding back the mean
    predictions = np.zeros_like(R_train)
    for u in range(n_users):
        u_id = unique_users[u]
        u_mean = train_user_means.get(u_id, global_mean)
        predictions[u, :] = predicted_deviations[u, :] + u_mean

    return predictions

mem_preds = predict_memory_based(R_train, train_user_means, global_train_mean)
